# **Machine Translation**

In [1]:
import tensorflow as tf
tf.keras.backend.clear_session()

In [2]:
import pandas as pd
from google.colab import files
uploaded = files.upload()
df=pd.read_csv("rus.txt", sep="\t", header=None, usecols=[0, 1], names=["english", "russian"])
print("first 5 rows")
print(df.head())
print("total sentance pairs")
print(len(df))

Saving rus.txt to rus.txt
first 5 rows
  english        russian
0     Go.          Марш!
1     Go.           Иди.
2     Go.         Идите.
3     Hi.  Здравствуйте.
4     Hi.        Привет!
total sentance pairs
363386


**Data preprocessing**

In [3]:
import re
import string
from sklearn.model_selection import train_test_split

file = "rus.txt"
pairs = []

with open(file, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            eng = parts[0]
            rus = parts[1]
            pairs.append((eng, rus))

print(pairs[:10])
print(len(pairs))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\d+", "", text)  # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()  # remove extra spaces
    return text

cleaned_pairs = []
for eng, rus in pairs:
    eng = clean_text(eng)
    rus = clean_text(rus)
    cleaned_pairs.append((eng, rus))

print(cleaned_pairs[:10])

[('Go.', 'Марш!'), ('Go.', 'Иди.'), ('Go.', 'Идите.'), ('Hi.', 'Здравствуйте.'), ('Hi.', 'Привет!'), ('Hi.', 'Хай.'), ('Hi.', 'Здрасте.'), ('Hi.', 'Здоро́во!'), ('Run!', 'Беги!'), ('Run!', 'Бегите!')]
363386
[('go', 'марш'), ('go', 'иди'), ('go', 'идите'), ('hi', 'здравствуйте'), ('hi', 'привет'), ('hi', 'хай'), ('hi', 'здрасте'), ('hi', 'здоро́во'), ('run', 'беги'), ('run', 'бегите')]


In [4]:
filtered_pairs = []

for eng, rus in cleaned_pairs:
    if len(eng.split()) <= 10 and len(rus.split()) <= 10:
        filtered_pairs.append((eng, rus))

print("Filtered pairs:", len(filtered_pairs))
print(filtered_pairs[:10])

Filtered pairs: 348679
[('go', 'марш'), ('go', 'иди'), ('go', 'идите'), ('hi', 'здравствуйте'), ('hi', 'привет'), ('hi', 'хай'), ('hi', 'здрасте'), ('hi', 'здоро́во'), ('run', 'беги'), ('run', 'бегите')]


In [5]:
processed_pairs = []

for eng, rus in filtered_pairs:
    rus = "<sos> " + rus + " <eos>"
    processed_pairs.append((eng, rus))

print(processed_pairs[:10])

[('go', '<sos> марш <eos>'), ('go', '<sos> иди <eos>'), ('go', '<sos> идите <eos>'), ('hi', '<sos> здравствуйте <eos>'), ('hi', '<sos> привет <eos>'), ('hi', '<sos> хай <eos>'), ('hi', '<sos> здрасте <eos>'), ('hi', '<sos> здоро́во <eos>'), ('run', '<sos> беги <eos>'), ('run', '<sos> бегите <eos>')]


In [6]:
MAX_SAMPLES = 100000
processed_pairs = processed_pairs[:MAX_SAMPLES]

print("Using pairs:", len(processed_pairs))
print(processed_pairs[:5])

Using pairs: 100000
[('go', '<sos> марш <eos>'), ('go', '<sos> иди <eos>'), ('go', '<sos> идите <eos>'), ('hi', '<sos> здравствуйте <eos>'), ('hi', '<sos> привет <eos>')]


In [7]:
from collections import Counter

def build_vocab(sentences, special_tokens, max_vocab_size=None):
    counter = Counter()

    for sentence in sentences:
        words = sentence.split()
        counter.update(words)

    vocab = {}
    for token in special_tokens:
        vocab[token] = len(vocab)

    if max_vocab_size is None:
        words_to_add = counter.items()
    else:
        words_to_add = counter.most_common(max_vocab_size)

    for item in words_to_add:
        word = item[0]
        if word not in vocab:
            vocab[word] = len(vocab)

    return vocab

In [8]:
english_sentences = []
russian_sentences = []

for eng, rus in processed_pairs:
    english_sentences.append(eng)
    russian_sentences.append(rus)

In [9]:
ENG_VOCAB_SIZE = 12000
RUS_VOCAB_SIZE = 16000

eng_vocab = build_vocab(
    english_sentences,
    ["<pad>", "<unk>"],
    max_vocab_size=ENG_VOCAB_SIZE
)

rus_vocab = build_vocab(
    russian_sentences,
    ["<pad>", "<unk>", "<sos>", "<eos>"],
    max_vocab_size=RUS_VOCAB_SIZE
)

print("English vocab size:", len(eng_vocab))
print("Russian vocab size:", len(rus_vocab))
print(list(eng_vocab.items())[:10])
print(list(rus_vocab.items())[:10])

English vocab size: 7218
Russian vocab size: 16002
[('<pad>', 0), ('<unk>', 1), ('tom', 2), ('i', 3), ('you', 4), ('is', 5), ('a', 6), ('it', 7), ('to', 8), ('the', 9)]
[('<pad>', 0), ('<unk>', 1), ('<sos>', 2), ('<eos>', 3), ('я', 4), ('том', 5), ('не', 6), ('это', 7), ('ты', 8), ('вы', 9)]


In [10]:
def sentence_to_indices(sentence, vocab):
    if isinstance(sentence, str):
        sentence = sentence.split()

    return [vocab.get(word, vocab["<unk>"]) for word in sentence]

In [11]:
data = []

for eng, rus in processed_pairs:
    eng_indices = sentence_to_indices(eng, eng_vocab)
    rus_indices = sentence_to_indices(rus, rus_vocab)
    data.append((eng_indices, rus_indices))

print(data[:10])

[([25], [2, 5642, 3]), ([25], [2, 223, 3]), ([25], [2, 331, 3]), ([982], [2, 4881, 3]), ([982], [2, 866, 3]), ([982], [2, 12115, 3]), ([982], [2, 12116, 3]), ([982], [2, 12117, 3]), ([331], [2, 3491, 3]), ([331], [2, 4882, 3])]


In [12]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

print("Train size:", len(train_data))
print("Validation size:", len(val_data))

print("Sample train:", train_data[0])
print("Sample val:", val_data[0])

Train size: 80000
Validation size: 20000
Sample train: ([3, 22, 6, 87, 1658], [2, 12, 15, 290, 2375, 3])
Sample val: ([3, 31, 20, 5009], [2, 10, 58, 346, 1, 3])


**Create data loaders**

In [13]:
train_src = []
train_tgt = []

for eng_indices, rus_indices in train_data:
    train_src.append(eng_indices)
    train_tgt.append(rus_indices)

val_src = []
val_tgt = []

for eng_indices, rus_indices in val_data:
    val_src.append(eng_indices)
    val_tgt.append(rus_indices)

print("Train examples:", len(train_src))
print("Validation examples:", len(val_src))

Train examples: 80000
Validation examples: 20000


In [14]:
train_src_padded = tf.keras.preprocessing.sequence.pad_sequences(
    train_src,
    padding="post",
    value=eng_vocab["<pad>"]
)

train_tgt_padded = tf.keras.preprocessing.sequence.pad_sequences(
    train_tgt,
    padding="post",
    value=rus_vocab["<pad>"]
)

val_src_padded = tf.keras.preprocessing.sequence.pad_sequences(
    val_src,
    padding="post",
    value=eng_vocab["<pad>"]
)

val_tgt_padded = tf.keras.preprocessing.sequence.pad_sequences(
    val_tgt,
    padding="post",
    value=rus_vocab["<pad>"]
)

print("Train source shape:", train_src_padded.shape)
print("Train target shape:", train_tgt_padded.shape)
print("Validation source shape:", val_src_padded.shape)
print("Validation target shape:", val_tgt_padded.shape)

Train source shape: (80000, 7)
Train target shape: (80000, 12)
Validation source shape: (20000, 7)
Validation target shape: (20000, 12)


In [15]:
# =========================================
# HYPERPARAMETERS
# =========================================
INPUT_DIM = len(eng_vocab)
OUTPUT_DIM = len(rus_vocab)
EMB_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 1
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 10

PAD_IDX = rus_vocab["<pad>"]
SOS_IDX = rus_vocab["<sos>"]
EOS_IDX = rus_vocab["<eos>"]

In [16]:
train_dataset = tf.data.Dataset.from_tensor_slices((train_src_padded, train_tgt_padded))
train_dataset = train_dataset.shuffle(len(train_src_padded)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_src_padded, val_tgt_padded))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

for src_batch, tgt_batch in train_dataset.take(1):
    print("Source batch shape:", src_batch.shape)
    print("Target batch shape:", tgt_batch.shape)

Source batch shape: (32, 7)
Target batch shape: (32, 12)


**Model architecture**

In [17]:
# =========================================
# MODEL ARCHITECTURE
# =========================================
import tensorflow as tf

class Encoder(tf.keras.Model):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(input_dim, emb_dim, mask_zero=True)
        self.lstm = tf.keras.layers.LSTM(
            hidden_dim,
            return_sequences=True,
            return_state=True
        )

    def call(self, x):
        x = self.embedding(x)
        outputs, hidden_state, cell_state = self.lstm(x)
        return outputs, hidden_state, cell_state


class Decoder(tf.keras.Model):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(output_dim, emb_dim, mask_zero=True)
        self.lstm = tf.keras.layers.LSTM(
            hidden_dim,
            return_sequences=True,
            return_state=True
        )
        self.fc = tf.keras.layers.Dense(output_dim)

    def call(self, x, hidden_state, cell_state):
        x = self.embedding(x)
        outputs, hidden_state, cell_state = self.lstm(
            x, initial_state=[hidden_state, cell_state]
        )
        predictions = self.fc(outputs)
        return predictions, hidden_state, cell_state


class Seq2Seq(tf.keras.Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def call(self, src, tgt, training=False, teacher_forcing_ratio=0.5):
        tgt_len = tgt.shape[1]

        _, hidden_state, cell_state = self.encoder(src)

        decoder_input = tf.expand_dims(tgt[:, 0], 1)   # <sos>
        outputs = []

        for t in range(1, tgt_len):
            predictions, hidden_state, cell_state = self.decoder(
                decoder_input, hidden_state, cell_state
            )

            outputs.append(predictions[:, 0, :])

            predicted_token = tf.argmax(
                predictions[:, 0, :], axis=-1, output_type=tf.int32
            )

            if training and tf.random.uniform(()) < teacher_forcing_ratio:
                next_input = tgt[:, t]
            else:
                next_input = predicted_token

            decoder_input = tf.expand_dims(next_input, 1)

        outputs = tf.stack(outputs, axis=1)   # (batch, tgt_len-1, vocab_size)
        return outputs

In [18]:
encoder = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM)
model = Seq2Seq(encoder, decoder)

print(model)

<Seq2Seq name=seq2_seq, built=False>


In [19]:
for src_batch, tgt_batch in train_dataset.take(1):
    outputs = model(src_batch, tgt_batch)
    print("Output shape:", outputs.shape)
    break

Output shape: (32, 11, 16002)


**Beam Search Decoding**

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
import os
import tensorflow as tf

SAVE_DIR = "/content/drive/MyDrive/seq2seq_rus_eng_model"

In [22]:
dummy_src = tf.constant([[1, 2, 3]], dtype=tf.int32)
dummy_tgt = tf.constant([[SOS_IDX, 4, 5, EOS_IDX]], dtype=tf.int32)
_ = model(dummy_src, dummy_tgt)

model.load_weights(os.path.join(SAVE_DIR, "model.weights.h5"))
print("Model loaded successfully")

Model loaded successfully


In [23]:
import math
import time
import heapq
from collections import Counter
import pandas as pd

idx_to_eng = {idx: word for word, idx in eng_vocab.items()}
idx_to_rus = {idx: word for word, idx in rus_vocab.items()}

def indices_to_english_sentence(indices):
    words = []
    for idx in indices:
        word = idx_to_eng.get(int(idx), "<unk>")
        if word == "<pad>":
            continue
        words.append(word)
    return " ".join(words)

def indices_to_russian_tokens(indices):
    tokens = []
    for idx in indices:
        word = idx_to_rus.get(int(idx), "<unk>")
        if word in ["<pad>", "<sos>"]:
            continue
        if word == "<eos>":
            break
        tokens.append(word)
    return tokens

In [24]:
def greedy_decode(sentence, max_len=20):
    sentence = clean_text(sentence)
    words = sentence.split()

    src_indices = [eng_vocab.get(word, eng_vocab["<unk>"]) for word in words]

    src_tensor = tf.keras.preprocessing.sequence.pad_sequences(
        [src_indices],
        padding="post"
    )
    src_tensor = tf.convert_to_tensor(src_tensor, dtype=tf.int32)

    _, hidden_state, cell_state = model.encoder(src_tensor)

    decoder_input = tf.constant([[SOS_IDX]], dtype=tf.int32)
    result_tokens = []

    for _ in range(max_len):
        predictions, hidden_state, cell_state = model.decoder(
            decoder_input, hidden_state, cell_state
        )

        predicted_id = int(tf.argmax(predictions[0, 0]).numpy())

        if predicted_id == EOS_IDX:
            break

        predicted_word = idx_to_rus.get(predicted_id, "<unk>")

        if predicted_word not in ["<sos>", "<eos>", "<pad>"]:
            result_tokens.append(predicted_word)

        decoder_input = tf.constant([[predicted_id]], dtype=tf.int32)

    return result_tokens

In [25]:
def beam_search(sentence, beam_width=5, max_len=20):
    sentence = clean_text(sentence)
    words = sentence.split()

    src_indices = [eng_vocab.get(word, eng_vocab["<unk>"]) for word in words]

    src_tensor = tf.keras.preprocessing.sequence.pad_sequences(
        [src_indices],
        padding="post"
    )
    src_tensor = tf.convert_to_tensor(src_tensor, dtype=tf.int32)

    _, hidden_state, cell_state = model.encoder(src_tensor)

    # Each beam item:
    # (score, token_ids, hidden_state, cell_state, finished)
    beams = [(0.0, [SOS_IDX], hidden_state, cell_state, False)]
    completed = []

    for _ in range(max_len):
        all_candidates = []

        for score, token_ids, h_state, c_state, finished in beams:
            if finished:
                all_candidates.append((score, token_ids, h_state, c_state, finished))
                continue

            last_token = token_ids[-1]
            decoder_input = tf.constant([[last_token]], dtype=tf.int32)

            predictions, new_h, new_c = model.decoder(
                decoder_input, h_state, c_state
            )

            log_probs = tf.nn.log_softmax(predictions[0, 0]).numpy()

            top_k_ids = log_probs.argsort()[-beam_width:][::-1]

            for token_id in top_k_ids:
                new_score = score + float(log_probs[token_id])
                new_token_ids = token_ids + [int(token_id)]
                new_finished = (int(token_id) == EOS_IDX)

                all_candidates.append(
                    (new_score, new_token_ids, new_h, new_c, new_finished)
                )

        # Keep only top beam_width candidates
        beams = sorted(all_candidates, key=lambda x: x[0], reverse=True)[:beam_width]

        # Save completed candidates
        completed.extend([beam for beam in beams if beam[4]])

        # If all beams are finished, stop early
        if all(beam[4] for beam in beams):
            break

    final_candidates = completed if completed else beams
    best_candidate = max(final_candidates, key=lambda x: x[0])

    best_token_ids = best_candidate[1]

    result_tokens = []
    for token_id in best_token_ids:
        if token_id in [SOS_IDX, EOS_IDX, PAD_IDX]:
            continue
        result_tokens.append(idx_to_rus.get(token_id, "<unk>"))

    return result_tokens

In [26]:
def get_ngrams(tokens, n):
    counts = Counter()
    for i in range(len(tokens) - n + 1):
        ngram = tuple(tokens[i:i+n])
        counts[ngram] += 1
    return counts

def clipped_precision(hypothesis, reference, n):
    hyp_ngrams = get_ngrams(hypothesis, n)
    ref_ngrams = get_ngrams(reference, n)

    total_hyp_ngrams = sum(hyp_ngrams.values())
    if total_hyp_ngrams == 0:
        return 0.0

    clipped_count = 0
    for ngram, hyp_count in hyp_ngrams.items():
        ref_count = ref_ngrams.get(ngram, 0)
        clipped_count += min(hyp_count, ref_count)

    return clipped_count / total_hyp_ngrams

def brevity_penalty(hypothesis, reference):
    c = len(hypothesis)
    r = len(reference)

    if c == 0:
        return 0.0
    if c > r:
        return 1.0
    return math.exp(1 - r / c)

def bleu_score(hypothesis, reference, max_n=4):
    precisions = []

    for n in range(1, max_n + 1):
        p_n = clipped_precision(hypothesis, reference, n)
        if p_n == 0:
            return 0.0
        precisions.append(p_n)

    log_precision_sum = sum((1 / max_n) * math.log(p) for p in precisions)
    bp = brevity_penalty(hypothesis, reference)

    return bp * math.exp(log_precision_sum)

In [27]:
example_sentence = "i am very tired"

print("SOURCE:", example_sentence)
print("GREEDY:", " ".join(greedy_decode(example_sentence)))
print("BEAM-3:", " ".join(beam_search(example_sentence, beam_width=3)))
print("BEAM-5:", " ".join(beam_search(example_sentence, beam_width=5)))
print("BEAM-10:", " ".join(beam_search(example_sentence, beam_width=10)))

SOURCE: i am very tired
GREEDY: я смертельно устала
BEAM-3: я смертельно устала
BEAM-5: я смертельно устала
BEAM-10: я смертельно устала


In [28]:
sample_val_data = val_data[:100]

results = []
example_rows = []

methods = [
    ("Greedy", None),
    ("Beam-3", 3),
    ("Beam-5", 5),
    ("Beam-10", 10),
]

for method_name, beam_width in methods:
    bleu4_scores = []
    total_time = 0.0

    for eng_indices, rus_indices in sample_val_data:
        eng_sentence = indices_to_english_sentence(eng_indices)
        ref_tokens = indices_to_russian_tokens(rus_indices)

        start_time = time.time()

        if beam_width is None:
            hyp_tokens = greedy_decode(eng_sentence)
        else:
            hyp_tokens = beam_search(eng_sentence, beam_width=beam_width)

        end_time = time.time()

        total_time += (end_time - start_time)
        bleu4_scores.append(bleu_score(hyp_tokens, ref_tokens, max_n=4))

    avg_bleu4 = sum(bleu4_scores) / len(bleu4_scores)
    avg_time = total_time / len(sample_val_data)

    results.append({
        "Method": method_name,
        "BLEU-4": avg_bleu4,
        "Avg Time Per Sentence (sec)": avg_time
    })

results_df = pd.DataFrame(results)
results_df

,Method,BLEU-4,Avg Time Per Sentence (sec)
0,Greedy,0.083961,0.070003
1,Beam-3,0.078187,0.142136
2,Beam-5,0.078187,0.206284
3,Beam-10,0.078187,0.405668


In [29]:
results_df["BLEU-4"] = results_df["BLEU-4"].round(4)
results_df["Avg Time Per Sentence (sec)"] = results_df["Avg Time Per Sentence (sec)"].round(4)
print(results_df.to_string(index=False))

 Method  BLEU-4  Avg Time Per Sentence (sec)
 Greedy  0.0840                       0.0700
 Beam-3  0.0782                       0.1421
 Beam-5  0.0782                       0.2063
Beam-10  0.0782                       0.4057


In [30]:
comparison_examples = []

for eng_indices, rus_indices in sample_val_data:
    eng_sentence = indices_to_english_sentence(eng_indices)
    ref_tokens = indices_to_russian_tokens(rus_indices)

    greedy_tokens = greedy_decode(eng_sentence)
    beam5_tokens = beam_search(eng_sentence, beam_width=5)

    greedy_bleu4 = bleu_score(greedy_tokens, ref_tokens, max_n=4)
    beam5_bleu4 = bleu_score(beam5_tokens, ref_tokens, max_n=4)

    comparison_examples.append({
        "EN": eng_sentence,
        "REF": " ".join(ref_tokens),
        "GREEDY": " ".join(greedy_tokens),
        "BEAM-5": " ".join(beam5_tokens),
        "GREEDY_BLEU4": greedy_bleu4,
        "BEAM5_BLEU4": beam5_bleu4,
        "DIFF": beam5_bleu4 - greedy_bleu4
    })

In [31]:
# Sort by where beam search helped the most
comparison_examples_sorted = sorted(
    comparison_examples,
    key=lambda x: x["DIFF"],
    reverse=True
)

top_5_examples = comparison_examples_sorted[:5]

for i, ex in enumerate(top_5_examples, 1):
    print(f"EXAMPLE {i}")
    print("EN     :", ex["EN"])
    print("REF    :", ex["REF"])
    print("GREEDY :", ex["GREEDY"])
    print("BEAM-5 :", ex["BEAM-5"])
    print("GREEDY BLEU-4 :", round(ex["GREEDY_BLEU4"], 4))
    print("BEAM-5 BLEU-4 :", round(ex["BEAM5_BLEU4"], 4))
    print("-" * 80)

EXAMPLE 1
EN     : i like your necklace
REF    : мне нравится ваше <unk>
GREEDY : мне нравится твоя <unk>
BEAM-5 : мне нравится твоя <unk>
GREEDY BLEU-4 : 0.0
BEAM-5 BLEU-4 : 0.0
--------------------------------------------------------------------------------
EXAMPLE 2
EN     : the grass looks nice
REF    : <unk> хорошо выглядит
GREEDY : пруд <unk> <unk>
BEAM-5 : пруд <unk> <unk>
GREEDY BLEU-4 : 0.0
BEAM-5 BLEU-4 : 0.0
--------------------------------------------------------------------------------
EXAMPLE 3
EN     : how is your dad
REF    : как твой папа
GREEDY : как ваш папа
BEAM-5 : как ваш папа
GREEDY BLEU-4 : 0.0
BEAM-5 BLEU-4 : 0.0
--------------------------------------------------------------------------------
EXAMPLE 4
EN     : i was very surprised
REF    : я очень удивился
GREEDY : я очень очень удивлена
BEAM-5 : я был очень удивлён
GREEDY BLEU-4 : 0.0
BEAM-5 BLEU-4 : 0.0
--------------------------------------------------------------------------------
EXAMPLE 5
EN     : somebo

In [33]:
RESULTS_DIR = "/content/drive/MyDrive/seq2seq_rus_eng_model"
os.makedirs(RESULTS_DIR, exist_ok=True)

results_df.to_csv(os.path.join(RESULTS_DIR, "beam_search_results.csv"), index=False)

examples_df = pd.DataFrame(top_5_examples)
examples_df.to_csv(os.path.join(RESULTS_DIR, "beam_search_examples.csv"), index=False)

print("Saved results to Drive.")

Saved results to Drive.
